In [0]:
from pyspark.sql.functions import col, to_date

df = spark.table("weather_project.bronze.hourly")

# Убираем  строки с Null 
df = df.filter(
    col("city_name").isNotNull() &
    col("datetime_valid_local").isNotNull()
)

# Нормализуем типы данных

df = df.withColumn("temperature", col("temperature").cast("Double")) \
        .withColumn("humidity_relative", col("humidity_relative").cast("Double"))

# Добавляем новые колонки
df = df.withColumn("event_date", to_date(col("datetime_valid_local"))) \
        .withColumn("is_rain_event", col("has_rain") | (col("rain_probability") > 50))


df.write.mode("overwrite").saveAsTable("weather_project.silver.hourly_clean")

# Считаем количество строк
hourly_clean_count = df.count()
print(f" Silver hourly: {hourly_clean_count} rows")

# Сохраняем значение hourly_clean_count  под ключом "hourly_clean_count, чтобы потом вызвать в 04_quality_checks
dbutils.jobs.taskValues.set(key="hourly_clean_count", value=hourly_clean_count)

display(spark.read.table("weather_project.silver.hourly_clean").limit(5))
